we can have one table for time, \
one table per group? one table per... 
need to decide later


In [1]:
import sys
sys.path.append('../../mdf_compression')

In [2]:
from mdfc.db import get_duckdb_buffer
test_fil = '/home/fish2/mdf_compression/examples/testing_duckdb.duckdb'
dd = get_duckdb_buffer(test_fil, overwrite=True)

In [8]:
dd.conn.close()

In [ ]:
from mdfc.transform.time import *
def compress_time(time_axis):
    ''' 
    take the unified time_axis (should be a f64 array)
    apply & record transformations in order
    return the compressed array (presently, u32 array)

    some values in time_axis are present with all other data from the MDF file
        --> which is, the original unified time values (float array, units of seconds)
    so it may be generated & saved outside of this function
    '''
    assert time_axis.dtype == np.float64, f"got time_axis with dtype {time_axis.dtype} but expected f64!"
    # capture ahead of time max value for error messaging
    maxval = time_axis.max()

    txs = []  # (enumeration, argument value) transformations applied, in order
    # we must make a copy of it :'( 
    #   or, reverse the transformations at the end? 
    time_axis = time_axis.copy()

    # scale up until reaching whole numbers
    #   or we exceed uint64 max range
    #   TODO we could go to uint128 max range but i dont want to yet
    scale = 1
    while not np.allclose(time_axis, np.round(time_axis)):
        scale *= 10
        time_axis = scale_up(time_axis, 10)  # just *=
        if time_axis.max() > MAX_U64_VALUE:
            # enhancement would be required
            raise ValueError(
                "File has timestamps that are too large & precise to be compressed... "
                f"scaleup factor needed to be {scale}, but file max timestamp of {maxval} "
                "would be out of range of uint64!")
    # record the transformation
    txs.append((TX_ENUMs[scale_up], scale))

    # convert f64 to u64
    time_axis = f64_to_u64(time_axis)
    txs.append((TX_ENUMs[f64_to_u64], ))

    # differentiate --> this implicitly casts u64 to f64 :'( big sad
    time_axis = diff(time_axis)
    txs.append((TX_ENUMs[diff], ))

    # ensure we dont exceed MAX_U32_VALUE in the differentiated axis
    # if so, it can be downcast & compressed
    if time_axis.max() > MAX_U32_VALUE:
        raise ValueError(
            "File has differential timestamps that exceed uint32 max value... "
            "Enhancement is required :'("
        )
    # else we can downcast to u32
    time_axis = u64_to_u32(time_axis)
    txs.append((TX_ENUMs[u64_to_u32], ))
    
    # compress!  and indicate if it was required to split 64 into 2*32
    #           this is always False for now... not supported yet
    # compressed, was_split = compress_u32(time_axis)
    # add placeholder to txs as the last thing
    # txs.append(was_split)
    return time_axis#, txs

In [ ]:
# lets see how much size the unified timeaxis is...
import asammdf
from mdfc.utils import unify_timestamps
# from io import BytesIO, numpy as np
# sample_data_path = '../sample_data/sample_data.mf4'
sample_data_path = '../sample_data/sample_data_high_random_values.mf4'
with (
    asammdf.MDF(sample_data_path) as mfil
):
    test_times = unify_timestamps(mfil)
    test_times = compress_time(test_times)

In [ ]:
test_times

In [ ]:
# how big does this become in ddb?
# dd.conn.executemany("INSERT INTO times (time) VALUES (?)", test_times)
# it might be better to use select so we dont have to tolist()?
dd.conn.execute('INSERT INTO times (time) SELECT * FROM test_times')
dd.conn.commit()
dd.conn.close()

In [ ]:
# testing read
import duckdb
test_fil = '/home/fish2/mdf_compression/examples/testing_duckdb.duckdb'
con = duckdb.connect(test_fil)

In [ ]:
# OK... so just putting times in one big array, as float,
#   causes 1 MB file size :(
# using integer compression library + metadata, 
#   we can get 20 kb size -> 0.02 MB 
# and on reading, it is much faster to read from the file
# than to select from duck
# so duck would make development eaiser
# but not as good compression
# and not as good "load everything" (slower)
#   possibly better "load down/up sampled"
#   better being memory better.. not better time wise

In [ ]:
# %%timeit
import numpy as np
np.asarray(con.execute('SElECT * FROM times').fetchall())

In [ ]:
from mdfc.compressor import MDFCompressor
from mdfc.decompressor import MDFDecompressor

In [ ]:
import asammdf
# from io import BytesIO, numpy as np
# sample_data_path = '../sample_data/sample_data.mf4'
sample_data_path = '../sample_data/sample_data_high_random_values.mf4'
testfil = '/home/fish2/mdf_compression/examples/testing_MDFCompressor.mdfc'

In [ ]:
with (
    asammdf.MDF(sample_data_path) as mfil,
    MDFCompressor(testfil, overwrite=True) as cfil
):
    cfil.unify_compress_time(mfil)
    test_times = cfil.time_axis  # it is saved to use in future signals compression
    cfil.finish()

In [ ]:
with MDFDecompressor(testfil) as dfil:
    dfil.decompress_time()
    confirm_times = dfil.time_axis

In [ ]:
# use allclose to avoid minor fp errors
import numpy as np
np.allclose(test_times, confirm_times)

In [ ]:
confirm_times

In [ ]:
test_times